In [1]:
"""
=====================================================================
 PROGRAM: Analisis Sifat Relasi (Refleksif, Simetris, Transitif,
          Proximity, dan Toleransi) pada Himpunan Fuzzy/Klasik
 Mata Kuliah : Fuzzy Terapan - Week 3 (Relasi & Komposisi)
=====================================================================

Program ini memeriksa apakah suatu relasi R (dinyatakan sebagai
matriks keanggotaan n x n) bersifat:
  1. Refleksif   : mu_R(xi, xi) = 1 untuk semua i
  2. Simetris    : mu_R(xi, xj) = mu_R(xj, xi) untuk semua i, j
  3. Transitif   : mu_R(xi, xk) >= min( mu_R(xi,xj), mu_R(xj,xk) )
                   untuk semua i, j, k
  4. Proximity   : refleksif DAN simetris (tetapi TIDAK transitif)
  5. Toleransi   : proximity, dan menjadi relasi ekuivalensi (transitif)
                   setelah paling banyak (n-1) kali komposisi max-min
                   terhadap dirinya sendiri (R^(n-1) = R o R o ... o R)

Jika relasi merupakan relasi toleransi, program akan menyebutkan
berapa kali komposisi (max-min) yang diperlukan agar relasi tersebut
menjadi transitif (relasi ekuivalensi).
"""

TOL = 1e-9  # toleransi numerik untuk perbandingan bilangan real


# ---------------------------------------------------------------
# 1. FUNGSI-FUNGSI DASAR PEMERIKSAAN SIFAT RELASI
# ---------------------------------------------------------------

def cek_refleksif(R):
    """Refleksif jika mu_R(xi, xi) = 1 untuk semua i."""
    n = len(R)
    return all(abs(R[i][i] - 1) <= TOL for i in range(n))


def cek_simetris(R):
    """Simetris jika mu_R(xi, xj) = mu_R(xj, xi) untuk semua i, j."""
    n = len(R)
    for i in range(n):
        for j in range(n):
            if abs(R[i][j] - R[j][i]) > TOL:
                return False
    return True


def cek_transitif(R):
    """
    Transitif (fuzzy max-min) jika untuk semua i, j, k:
        mu_R(xi, xk) >= min( mu_R(xi,xj), mu_R(xj,xk) )
    Mengembalikan (True/False, daftar pelanggaran).
    Catatan: untuk relasi klasik (0/1), rumus ini otomatis tereduksi
    menjadi definisi transitif klasik.
    """
    n = len(R)
    pelanggaran = []
    for i in range(n):
        for j in range(n):
            for k in range(n):
                batas_bawah = min(R[i][j], R[j][k])
                if R[i][k] < batas_bawah - TOL:
                    pelanggaran.append((i + 1, j + 1, k + 1, R[i][k], batas_bawah))
    return (len(pelanggaran) == 0), pelanggaran


def komposisi_max_min(A, B):
    """Komposisi max-min: C = A o B, C(i,j) = max_k min(A(i,k), B(k,j))."""
    n = len(A)
    p = len(B)
    m = len(B[0])
    C = [[0.0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            C[i][j] = max(min(A[i][k], B[k][j]) for k in range(p))
    return C


def cek_proximity(R):
    """
    Relasi proximity = refleksif DAN simetris, TETAPI TIDAK transitif
    (mengikuti definisi pada slide: relasi proximity adalah relasi
    refleksif dan simetris, namun tidak transitif).
    """
    refl = cek_refleksif(R)
    sim = cek_simetris(R)
    trans, _ = cek_transitif(R)
    return refl and sim and (not trans)


def cek_toleransi(R):
    """
    Relasi toleransi: proximity, dan menjadi transitif (ekuivalensi)
    melalui paling banyak (n-1) kali komposisi max-min dengan dirinya
    sendiri: R^(n-1) = R o R o ... o R.

    Mengembalikan dict dengan info:
        - is_toleransi (bool)
        - jumlah_komposisi (int atau None) -> banyaknya operasi 'o'
          yang diperlukan sampai transitif tercapai
        - matriks_akhir -> matriks R^k yang sudah transitif (jika ada)
    """
    n = len(R)
    refl = cek_refleksif(R)
    sim = cek_simetris(R)

    if not (refl and sim):
        return {
            "is_toleransi": False,
            "jumlah_komposisi": None,
            "alasan": "Bukan relasi proximity (tidak refleksif dan/atau tidak simetris), "
                      "sehingga tidak dapat berupa relasi toleransi."
        }

    # Jika R sendiri sudah transitif -> R adalah relasi ekuivalensi
    # (kasus khusus, 0 kali komposisi tambahan diperlukan)
    trans, _ = cek_transitif(R)
    if trans:
        return {
            "is_toleransi": True,
            "jumlah_komposisi": 0,
            "matriks_akhir": R,
            "keterangan": "R sudah transitif tanpa komposisi tambahan (R adalah relasi ekuivalensi)."
        }

    Rk = R
    maks_komposisi = n - 1
    for k in range(1, maks_komposisi + 1):
        Rk = komposisi_max_min(Rk, R)   # R^k = R^(k-1) o R
        trans, _ = cek_transitif(Rk)
        if trans:
            return {
                "is_toleransi": True,
                "jumlah_komposisi": k,
                "matriks_akhir": Rk,
                "keterangan": f"R^{k} = R komposisi (max-min) sebanyak {k} kali sudah transitif."
            }

    return {
        "is_toleransi": False,
        "jumlah_komposisi": None,
        "alasan": f"Relasi R merupakan proximity, namun tetap tidak transitif "
                  f"meski sudah dikomposisikan sebanyak (n-1) = {maks_komposisi} kali. "
                  f"Berarti R BUKAN relasi toleransi."
    }


# ---------------------------------------------------------------
# 2. FUNGSI UTILITAS: CETAK MATRIKS & LAPORAN
# ---------------------------------------------------------------

def cetak_matriks(R, judul=None):
    if judul:
        print(judul)
    for baris in R:
        print("  [" + "  ".join(f"{v:.2f}" for v in baris) + "]")
    print()


def laporan_relasi(R, nama="R"):
    print("=" * 70)
    print(f"ANALISIS RELASI: {nama}")
    print("=" * 70)
    cetak_matriks(R, f"Matriks {nama}:")

    refl = cek_refleksif(R)
    sim = cek_simetris(R)
    trans, pelanggaran = cek_transitif(R)
    prox = cek_proximity(R)

    print(f"1. Refleksif : {'YA' if refl else 'TIDAK'}")
    print(f"2. Simetris  : {'YA' if sim else 'TIDAK'}")
    print(f"3. Transitif : {'YA' if trans else 'TIDAK'}")
    if not trans and pelanggaran:
        i, j, k, nilai, batas = pelanggaran[0]
        print(f"   -> Pelanggaran contoh: mu(x{i},x{k})={nilai:.2f} "
              f"< min(mu(x{i},x{j}), mu(x{j},x{k}))={batas:.2f}")
    print(f"4. Proximity : {'YA' if prox else 'TIDAK'}")

    if refl and sim and trans:
        print("   -> R sudah transitif sejak awal, sehingga R adalah RELASI EKUIVALENSI.")

    hasil_tol = cek_toleransi(R)
    if hasil_tol["is_toleransi"]:
        k = hasil_tol["jumlah_komposisi"]
        print(f"5. Toleransi : YA")
        if k == 0:
            print("   -> R sudah transitif tanpa perlu komposisi tambahan.")
        else:
            print(f"   -> Diperlukan {k} kali komposisi max-min (R^{k} = R o R o ... o R, "
                  f"{k} kali) agar R menjadi transitif (relasi ekuivalensi).")
            cetak_matriks(hasil_tol["matriks_akhir"], f"   Matriks R^{k} (sudah transitif):")
    else:
        print(f"5. Toleransi : TIDAK  ({hasil_tol['alasan']})")
    print()


# ---------------------------------------------------------------
# 3. CONTOH PENGGUNAAN: SOAL-SOAL PADA MODUL WEEK 3
# ---------------------------------------------------------------

if __name__ == "__main__":

    # --- Contoh pada slide (5x5), butuh 2 kali komposisi ---
    Contoh = [
        [1,   0.6, 0,   0.3, 0.2],
        [0.6, 1,   0.5, 0.8, 0],
        [0,   0.5, 1,   0,   0.4],
        [0.3, 0.8, 0,   1,   0.5],
        [0.2, 0,   0.4, 0.5, 1],
    ]
    laporan_relasi(Contoh, "Contoh (slide)")

    # --- Latihan 1 (relasi klasik 0/1) ---
    Latihan1 = [
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 1],
        [0, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
        [0, 1, 1, 0, 1],
    ]
    laporan_relasi(Latihan1, "Latihan 1")

    # --- Latihan 2 ---
    Latihan2 = [
        [1,   0.8, 0.4, 0.5, 0.6],
        [0.5, 1,   0.4, 0.3, 0.9],
        [0.4, 0.2, 1,   0.4, 0.4],
        [0.5, 0.5, 0.3, 1,   0.3],
        [0.4, 0.9, 0.4, 0.7, 1],
    ]
    laporan_relasi(Latihan2, "Latihan 2")

    # --- Latihan 3 ---
    Latihan3 = [
        [1,   0.6, 0,   0.2, 0.3],
        [0.6, 1,   0.5, 0,   0.8],
        [0,   0.5, 1,   0,   0],
        [0.2, 0,   0,   1,   0.5],
        [0.3, 0.8, 0,   0.5, 1],
    ]
    laporan_relasi(Latihan3, "Latihan 3")

    # --- Latihan 4 (6x6) ---
    Latihan4 = [
        [1,   0.2, 1,   0.6, 0.2, 0.6],
        [0.2, 1,   0.2, 0.2, 0.8, 0.2],
        [1,   0.2, 1,   0.6, 0.2, 0.6],
        [0.6, 0.2, 0.6, 1,   0.2, 0.8],
        [0.2, 0.8, 0.2, 0.2, 1,   0.2],
        [0.6, 0.2, 0.6, 0.8, 0.2, 1],
    ]
    laporan_relasi(Latihan4, "Latihan 4")

ANALISIS RELASI: Contoh (slide)
Matriks Contoh (slide):
  [1.00  0.60  0.00  0.30  0.20]
  [0.60  1.00  0.50  0.80  0.00]
  [0.00  0.50  1.00  0.00  0.40]
  [0.30  0.80  0.00  1.00  0.50]
  [0.20  0.00  0.40  0.50  1.00]

1. Refleksif : YA
2. Simetris  : YA
3. Transitif : TIDAK
   -> Pelanggaran contoh: mu(x1,x3)=0.00 < min(mu(x1,x2), mu(x2,x3))=0.50
4. Proximity : YA
5. Toleransi : YA
   -> Diperlukan 2 kali komposisi max-min (R^2 = R o R o ... o R, 2 kali) agar R menjadi transitif (relasi ekuivalensi).
   Matriks R^2 (sudah transitif):
  [1.00  0.60  0.50  0.60  0.50]
  [0.60  1.00  0.50  0.80  0.50]
  [0.50  0.50  1.00  0.50  0.50]
  [0.60  0.80  0.50  1.00  0.50]
  [0.50  0.50  0.50  0.50  1.00]


ANALISIS RELASI: Latihan 1
Matriks Latihan 1:
  [1.00  1.00  0.00  0.00  0.00]
  [1.00  1.00  1.00  0.00  1.00]
  [0.00  0.00  1.00  0.00  0.00]
  [0.00  1.00  0.00  1.00  0.00]
  [0.00  1.00  1.00  0.00  1.00]

1. Refleksif : YA
2. Simetris  : TIDAK
3. Transitif : TIDAK
   -> Pelanggaran